In [ ]:
# from collections import Counter
# from sklearn.datasets import make_classification
# from imblearn.under_sampling import RandomUnderSampler
# X, y = make_classification(n_classes=2, class_sep=2,
#  weights=[0.1, 0.9], n_informative=3, n_redundant=1, flip_y=0,
# n_features=20, n_clusters_per_class=1, n_samples=1000, random_state=10)
# print('Original dataset shape %s' % Counter(y))
# rus = RandomUnderSampler(random_state=42)
# X_res, y_res = rus.fit_resample(X, y)
# print('Resampled dataset shape %s' % Counter(y_res))


In [6]:
from pathlib import Path
import sys
from collections import Counter
from dataclasses import asdict

import pandas as pd
import yaml

repo_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = None
for candidate in repo_candidates:
    if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    REPO_ROOT = Path("/home/users/roeizucker/tests/jupyter_notebooks/Tom_Hope_Project/refactored_code")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.constants import INSTADEEP_KMER_SIZE
from src.token_classification_data_extractor import _load_token_dataset
from src.utils.token_label_binning_utils import (
    apply_token_label_binning_to_dataset,
    resolve_token_label_binning,
)

print("repo root:", REPO_ROOT)


repo root: /sci/nosnap/michall/roeizucker/jupyter_notebooks/Tom_Hope_Project/refactored_code


In [8]:
CONFIG_PATH = REPO_ROOT / "configs" / "example_token_classification_extraction.yaml"
MAX_INTERMEDIATE_ROWS = 500  # keep notebook iteration fast; set to None to use the full intermediate CSV

cfg = yaml.safe_load(CONFIG_PATH.read_text())
task = cfg["task"]
paths = cfg["paths"]

source_train_intermediate_path = Path(paths["intermediate_train_data_path"])
if not source_train_intermediate_path.exists():
    raise FileNotFoundError(
        f"Missing intermediate train CSV: {source_train_intermediate_path}. "
        "Run extraction first, or point CONFIG_PATH to a config whose intermediate file exists."
    )

train_intermediate_path = source_train_intermediate_path
if MAX_INTERMEDIATE_ROWS is not None:
    train_intermediate_path = Path("/tmp") / f"{source_train_intermediate_path.stem}_first_{MAX_INTERMEDIATE_ROWS}.csv"
    pd.read_csv(source_train_intermediate_path, nrows=MAX_INTERMEDIATE_ROWS).to_csv(train_intermediate_path, index=False)

train_ds = _load_token_dataset(
    task,
    str(train_intermediate_path),
    task.get("kmer_size", INSTADEEP_KMER_SIZE),
    task["seq_size"],
)

print(train_ds)
print("columns:", train_ds.column_names)
print("first labels preview:", train_ds[0]["labels"][:20])


Dataset({
    features: ['seq', 'labels', 'start', 'end', 'window_id'],
    num_rows: 497
})
columns: ['seq', 'labels', 'start', 'end', 'window_id']
first labels preview: [-100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0]


In [9]:
resolved_binning = resolve_token_label_binning(task, train_ds)
train_ds = apply_token_label_binning_to_dataset(train_ds, resolved_binning)


def count_non_blank_token_labels(dataset, label_column="labels", blank_label=-100):
    counts = Counter()
    for example in dataset:
        for label in example[label_column]:
            label = int(label)
            if label != blank_label:
                counts[label] += 1
    return dict(sorted(counts.items()))


print("resolved binning:", asdict(resolved_binning))
print("binned train label counts:", count_non_blank_token_labels(train_ds, blank_label=resolved_binning.blank_label))
print("first binned labels preview:", train_ds[0]["labels"][:20])

# Next step, once we implement it together:
# from src.utils.simplefied_downsampling_utils import apply_token_label_downsampling_to_dataset
# train_ds = apply_token_label_downsampling_to_dataset(train_ds, task, seed=cfg.get("random_state", 42))


resolved binning: {'method': 'fixed', 'low': 0.2, 'high': 0.8, 'resolved_low': 0.2, 'resolved_high': 0.8, 'blank_label': -100}
binned train label counts: {0: 1591, 1: 6441, 2: 12927}
first binned labels preview: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]


In [5]:
# ratio = 0.5
# minority_class = -1
# minority_count = None
# dic = count_non_blank_token_labels(train_ds, blank_label=resolved_binning.blank_label)
# for key in dic:
#     if minority_count is None or dic[key] < minority_count:
#         minority_count = dic[key]
#         minority_class = key
# print(minority_count)

In [10]:
from imblearn.under_sampling import RandomUnderSampler
import numpy as np


In [ ]:
from src.utils.simplefied_downsampling_utils import downsample_dataset

ratio = 1
dataset_to_resample = train_ds
blank_label = -100
random_state = 42


downsampled_train_ds = downsample_dataset(ratio, dataset_to_resample, blank_label, random_state)


{0: 1591, 1: 1591, 2: 1591}

{0: 1591, 1: 3182, 2: 3182}